<a href="https://colab.research.google.com/github/faguilarleal/lab10_vision/blob/main/lab10_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install diffusers transformers accelerate xformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 23.3 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

# Verificar disponibilidad de GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # reduce VRAM usage

print("Modelo cargado correctamente.")

    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.10.0+cpu)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Usando dispositivo: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure 

Modelo cargado correctamente.


In [3]:
PROMPT = (
    "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, "
    "neon lights, 4k resolution"
)
SEED           = 42
NUM_STEPS      = 20
CAPTURE_STEPS  = {4, 10, 16, 20}   # pasos (1-indexed) que queremos guardar

saved_latents: dict[int, torch.Tensor] = {}

def callback_on_step_end(pipeline, step_index: int, timestep, callback_kwargs: dict):
    """
    Intercepta el tensor latente al final de cada paso de denoising.
    step_index es 0-indexed, por lo que el paso 1 corresponde a step_index=0.
    """
    current_step = step_index + 1          # convertir a 1-indexed
    if current_step in CAPTURE_STEPS:
        # .clone() para evitar que el pipeline sobreescriba el tensor in-place
        saved_latents[current_step] = callback_kwargs["latents"].clone()
        print(f"  -> Latente capturado en el paso {current_step}/{NUM_STEPS}  "
              f"shape={callback_kwargs['latents'].shape}  dtype={callback_kwargs['latents'].dtype}")
    return callback_kwargs                 # devolver sin modificar

print("Callback definido. Listo para ejecutar el pipeline.")

Callback definido. Listo para ejecutar el pipeline.


In [ ]:
torch.manual_seed(SEED)

output = pipe(
    prompt=PROMPT,
    num_inference_steps=NUM_STEPS,
    callback_on_step_end=callback_on_step_end,
    callback_on_step_end_inputs=["latents"],  # qué tensores inyectar en callback_kwargs
)

final_image = output.images[0]
final_image.save("final_image.png")
print(f"\nPipeline completado. Latentes capturados en los pasos: {sorted(saved_latents.keys())}")
final_image

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
scaling_factor = pipe.vae.config.scaling_factor
print(f"VAE scaling_factor: {scaling_factor}")

decoded_images: dict[int, Image.Image] = {}

def latent_to_pil(latent: torch.Tensor) -> Image.Image:
    """Decodifica un tensor latente (1,4,H,W) a imagen PIL."""
    # 1. Revertir el escalado interno del pipeline
    z = latent / scaling_factor

    # 2. Pasar por el decoder del VAE (sin gradientes para ahorrar memoria)
    with torch.no_grad():
        decoded = pipe.vae.decode(z).sample   # shape: (1, 3, H, W)  rango [-1, 1]

    # 3. Mapear de [-1,1] a [0,1] y recortar por seguridad
    decoded = (decoded / 2 + 0.5).clamp(0, 1)

    # 4. Convertir a numpy uint8
    decoded_np = decoded.cpu().float().permute(0, 2, 3, 1).numpy()  # (1,H,W,3)
    img_array  = (decoded_np[0] * 255).round().astype(np.uint8)

    return Image.fromarray(img_array)

for step in sorted(saved_latents.keys()):
    img = latent_to_pil(saved_latents[step])
    decoded_images[step] = img
    filename = f"latent_step_{step:02d}.png"
    img.save(filename)
    print(f"Paso {step:2d}/{NUM_STEPS} -> guardado como '{filename}'  tamaño={img.size}")

print("\nDecodificación completada.")

In [ ]:
steps_to_show = sorted(decoded_images.keys())   # [4, 10, 16, 20]
n_cols = len(steps_to_show) + 1                 # +1 para la imagen final del pipeline

fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
fig.suptitle(
    "Evolución del Espacio Latente durante el Denoising\n"
    f'Prompt: "{PROMPT[:60]}…"',
    fontsize=11, y=1.02
)

for ax, step in zip(axes[:-1], steps_to_show):
    ax.imshow(decoded_images[step])
    ax.set_title(f"Paso {step}/{NUM_STEPS}\n(VAE decode)")
    ax.axis("off")

# Última columna: imagen final generada por el pipeline completo
axes[-1].imshow(final_image)
axes[-1].set_title(f"Imagen final\n(pipeline completo)")
axes[-1].axis("off")

plt.tight_layout()
plt.savefig("diffusion_latent_evolution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada como 'diffusion_latent_evolution.png'")